In [31]:
df = pl.scan_csv("/Users/junhan/song-recommender/song_lyrics.csv") # lazyframe
print(df.schema)

Schema([('title', String), ('tag', String), ('artist', String), ('year', Int64), ('views', Int64), ('features', String), ('lyrics', String), ('id', Int64), ('language_cld3', String), ('language_ft', String), ('language', String)])


/var/folders/ks/zrg3bwdn7598g9tvjjz3h7900000gn/T/ipykernel_55736/1967306008.py:2: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print(df.schema)


In [ ]:
query = (
    df.select(['title','tag','artist','lyrics','language_cld3'])
    .filter(pl.col('language_cld3').is_in(['ja','zh','nan']))
)

result = query.collect(streaming=True)

/var/folders/ks/zrg3bwdn7598g9tvjjz3h7900000gn/T/ipykernel_51896/83184733.py:6: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  result = query.collect(streaming=True)


In [32]:
print(result.head)
print(result.shape)
result.write_parquet("/Users/junhan/song-recommender/song_lyrics.parquet")

NameError: name 'result' is not defined

In [34]:
import polars as pl

df = pl.read_parquet("/Users/junhan/song-recommender/song_lyrics.parquet")
#print(df.filter(pl.col("artist").str.contains("(?i)Blue")))

In [35]:
from langchain_community.document_loaders import PolarsDataFrameLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [36]:
import torch
print(torch.backends.mps.is_available())

True


In [37]:
model_name = "BAAI/bge-m3"
model_kwargs = {"device" : "mps"}
encode_kwargs = {"normalize_embeddings": True}
hf = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs)


In [38]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, 
    chunk_overlap=50
)

vector_db = Chroma(collection_name="songs", embedding_function=hf, persist_directory="./lyric_database")

In [39]:
import itertools
from tqdm import tqdm

In [41]:
loader = PolarsDataFrameLoader(df, page_content_column="lyrics")
docs = loader.lazy_load()

def batch_iterator(docs, size):
    it = iter(docs)
    while True: 
        chunk = list(itertools.islice(it, size))
        if not chunk:
            break
        yield chunk

for batch in tqdm(batch_iterator(docs,500), total=134):
    split_docs = splitter.split_documents(batch)
    vector_db.add_documents(split_docs)

  1%|▏         | 2/134 [14:44<16:12:50, 442.20s/it]


KeyboardInterrupt: 